<a href="https://colab.research.google.com/github/elhamod/IS883_Fall_2026/blob/main/Session%2005/IS883_Session05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS883 Session 5: Giving LLMs Tools and Knowledge

Last week we saw two big limits of an LLM on its own:
- it can't **do** anything (look up a date, run exact math, hit an API), and
- it only knows what was in its training data — it can't see **your** documents.

This week we fix both:
1. **Tools (function calling)** — let the model call your Python functions, use Gemini's built-in tools, and reach tools on an **MCP server**.
2. **Chain-of-Thought vs. ReAct** — reasoning in steps vs. reasoning *and acting* in steps, with every Thought -> Action -> Observation step printed so you can follow it.
3. **Embeddings** — turn text into numbers that capture meaning.
4. **Chunking** — split long documents into pieces, and see why it matters.
5. **Retrieval with cosine similarity** — find the most relevant chunks for a question (using scikit-learn).
6. **RAG** — combine retrieval + generation so the model answers from *your* knowledge.

# Part 0: Setup

In [ ]:
# Install Google's GenAI SDK for the Gemini API, yfinance for stock prices (Part 2), and the MCP library.
!pip install -q google-genai yfinance mcp

**Where the API key comes from.** Never paste a key into a notebook — anyone you share the file with gets your key, and a key pushed to a public repo is found within minutes. Instead the helper below looks for it in three places, in order:

1. **Colab Secrets** (the 🔑 panel on the left) — this is how you should do it in this course.
2. An **environment variable** of the same name — for VS Code or a local Jupyter.
3. Failing both, it **asks you to paste it**, and even then the value never lands in the file.

In [ ]:
# Read the API key from wherever this notebook is running, without writing it into the file.
import os, getpass

def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass.getpass(f"{name}: ")

In [ ]:
# Connect to the Gemini API using the key you saved as a Colab secret (named MyGeminiKey).
from google import genai
from google.genai import types

client = genai.Client(api_key=get_key('MyGeminiKey'))

MODEL = "gemini-3.1-flash-lite"

In [ ]:
# Helper: send ONE request to Gemini and return its text, waiting and retrying if the API is busy.
import time

# Pick the model with model=... . The default is MODEL (gemini-3.1-flash-lite); other free-tier text models:
#   model="gemini-3.5-flash-lite"   newer Flash-Lite, the natural upgrade
#   model="gemini-3.6-flash"        full Flash: stronger at reasoning and tool use
#   model="gemini-3.8-flash"        newest full Flash
#   model="gemini-3-flash-preview"  preview model; may change without notice
#   model="gemini-2.5-flash"        older generation (2.5 models may reject the thinking setting below)
#   model="gemini-2.5-flash-lite"   older generation, smallest
#   model="gemini-2.5-pro"          older generation; the only free "Pro" model, with tight limits
def ask_gemini(prompt, model=MODEL, system_instruction=None, seed=None, temperature=None,
               max_output_tokens=None, stop_sequences=None, response_schema=None,
               show_usage=False, thinking=False, include_thoughts=False,
               tools=None, auto_call_tools=True, full_response=False, show_messages=False, verbose=False):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        seed=seed,
        temperature=temperature,
        max_output_tokens=max_output_tokens,
        stop_sequences=stop_sequences,
        # Only ask for JSON when we pass a schema (the "form" the answer must fill in).
        response_mime_type="application/json" if response_schema else None,
        response_schema=response_schema,
        # thinking=False keeps the model's built-in "thinking" minimal so we see the effect of OUR prompts.
        thinking_config=types.ThinkingConfig(thinking_level="HIGH" if thinking else "MINIMAL", include_thoughts=include_thoughts),
        # Python functions the model may call. auto_call_tools=False lets US run them (Part 2).
        tools=tools,
        automatic_function_calling=None if auto_call_tools else types.AutomaticFunctionCallingConfig(disable=True),
    )

    # With tools, the SDK quietly sends Gemini MORE THAN ONE request (it runs your function and sends the
    # result back). To count them all, we listen in on each request it sends. (This relies on an internal
    # SDK method, _generate_content; the SDK itself still does all the work.)
    usages = []
    send_one_request = client.models._generate_content
    def count_request(**kwargs):
        reply = send_one_request(**kwargs)
        usages.append(reply.usage_metadata)
        if show_messages:
            print_messages(len(usages), kwargs["contents"], kwargs["config"], reply)
        return reply

    # show_messages=True prints, for each request, everything sent to Gemini and everything it sent back.
    # verbose=True also prints the settings and each tool's full description; otherwise just their names.
    def print_turn(role, part):
        if part.text:
            label = "thinking" if part.thought else role
            print(f"  [{label}] {part.text[:800]}")
        if part.function_call:
            print(f"  [{role} asks to run] {part.function_call.name}({dict(part.function_call.args)})")
        if part.function_response:
            print(f"  [{role} sends result] {part.function_response.name} -> {part.function_response.response}")
        if part.executable_code:
            print(f"  [{role} wrote code]\n{part.executable_code.code}")
        if part.code_execution_result:
            print(f"  [code output] {part.code_execution_result.output}")

    def print_messages(number, contents, sent_config, reply):
        print(f"===== request {number}: SENT to Gemini =====")
        if sent_config and sent_config.system_instruction:
            print(f"  [system instruction] {sent_config.system_instruction}")
        if sent_config and verbose:
            # The settings sent with the request (temperature, thinking, JSON schema, ...).
            try:
                settings = sent_config.model_dump_json(exclude_none=True, indent=2,
                                                       exclude={"tools", "system_instruction", "automatic_function_calling", "http_options"})
            except Exception:
                settings = str(sent_config)
            print(f"  [settings] {settings}")
        described = []
        for tool in (sent_config.tools or []) if sent_config else []:
            # Each tool's full description, exactly as Gemini receives it: name, docstring, argument types.
            if callable(tool):
                tool = types.FunctionDeclaration.from_callable(client=client, callable=tool)
            if verbose:
                print(f"  [tool offered] {tool.model_dump_json(exclude_none=True, indent=2)}")
            else:
                described.append(getattr(tool, "name", None) or ", ".join(tool.model_dump(exclude_none=True)))
        if described:
            print("  [tools offered]", ", ".join(described), "(verbose=True shows their full descriptions)")
        for turn in (contents if isinstance(contents, list) else [contents]):
            if isinstance(turn, str):
                print(f"  [user] {turn[:800]}")
            else:
                for part in turn.parts or []:
                    print_turn(turn.role or "user", part)
        print(f"===== request {number}: RECEIVED from Gemini =====")
        for part in reply.candidates[0].content.parts or []:
            print_turn("model", part)
        print()

    # Retry a few times: the free tier's per-minute limit (429), a briefly overloaded
    # model (503) and a timeout (504) all stop a call that is otherwise fine, and all pass.
    for attempt in range(4):
        usages.clear()
        client.models._generate_content = count_request
        try:
            response = client.models.generate_content(model=model, contents=prompt, config=config)
        except Exception as error:
            message = str(error) + type(error).__name__
            if "429" in message or "RESOURCE_EXHAUSTED" in message:
                print("   (free-tier rate limit reached, waiting 15 seconds...)")
            elif "503" in message or "UNAVAILABLE" in message:
                print("   (model overloaded, waiting 15 seconds...)")
            elif "504" in message or "DEADLINE_EXCEEDED" in message or "Timeout" in message:
                print("   (request timed out, waiting 15 seconds...)")
            else:
                raise
            time.sleep(15)
            continue
        finally:
            client.models._generate_content = send_one_request

        # Show how many tokens each request used, plus the total when there was more than one
        # (this is what an API bills for).
        if show_usage:
            rows = [("input_tokens", "prompt_token_count"), ("output_tokens", "candidates_token_count"),
                    ("thinking_tokens", "thoughts_token_count"), ("tools_tokens", "tool_use_prompt_token_count"),
                    ("cached_tokens", "cached_content_token_count"), ("total_tokens", "total_token_count")]
            headers = [f"request {n}" for n in range(1, len(usages) + 1)] + (["TOTAL"] if len(usages) > 1 else [])
            print("*" * 20)
            print(" " * 17 + "".join(f"{h:>12}" for h in headers))
            for label, field in rows:
                values = [getattr(usage, field) or 0 for usage in usages]
                if len(values) > 1:
                    values.append(sum(values))
                print(f"{label + ':':17}" + "".join(f"{v:>12}" for v in values))
            print("*" * 20)

        # If asked, print the model's own reasoning (a summary of its "thoughts") before the answer.
        if include_thoughts:
            for part in response.candidates[0].content.parts:
                if part.thought and part.text:
                    print("----- MODEL'S THINKING (summary) -----")
                    print(part.text)
                    print("----- END OF THINKING, ANSWER BELOW -----")

        # full_response=True hands back the whole response, so we can see which tools were called.
        if full_response:
            return response

        return response.text or "[the model returned no text]"

    # Every retry failed: say what to try instead of failing silently.
    print("Gemini is still busy or unavailable after 4 tries. Things to try:")
    print('  1. Switch models for this call, e.g. ask_gemini(prompt, model="gemini-3.5-flash-lite").')
    print("     Each model has its own free-tier limit, so another model usually works right away.")
    print("  2. Wait a minute and re-run the cell: the per-minute limit resets quickly.")
    print("  3. Hit the DAILY limit? It resets at midnight Pacific time; switch models until then.")
    print("  4. Still stuck? Let a teammate run the notebook with their own API key.")
    if full_response:
        raise RuntimeError("Gemini did not respond. See the suggestions above.")
    return "[gave up after repeated retries]"

# Part 1: Tools (Function Calling)

An LLM predicts text; it is **not** a calculator, a clock, or a database. Ask it for exact arithmetic and it may quietly get it wrong. The fix is to hand it **tools**: ordinary Python functions it can choose to call. Let's first watch it struggle without one.

In [ ]:
# Exact arithmetic is a classic weak spot. Ask and check the number carefully.
question = ("I have $110,345.45 in savings at 6.46% annual interest, compounded yearly. "
            "How much will I have after exactly one year? Give the exact number.")
print(ask_gemini(question, show_usage=True))

Now we define two **tools**. A tool is just a normal function with a clear name, typed arguments, and a docstring — Gemini reads the docstring to decide *when* to use it.

In [ ]:
# Two tools. Gemini reads each docstring to decide whether and how to call it.
from datetime import date

def compound_interest(principal: float, annual_rate_percent: float, years: float) -> float:
    """Return the final amount for money growing at a fixed annual interest rate.

    Args:
        principal: the starting amount of money.
        annual_rate_percent: the yearly interest rate as a percent (e.g. 6.46 means 6.46%).
        years: how many years the money grows.
    """
    return principal * (1 + annual_rate_percent / 100) ** years

def get_today() -> str:
    """Return today's date as a YYYY-MM-DD string."""
    return date.today().isoformat()

### Automatic function calling
The simplest way: pass the functions as `tools` and let the SDK run the whole loop. Gemini decides which tool to call, calls it, reads the result, and gives you the final answer.

In [ ]:
# Give Gemini the tools and let it use them automatically; show_messages reveals each request behind the scenes.
print(ask_gemini(question, tools=[compound_interest, get_today], show_usage=True, show_messages=True, verbose=True))

### A tool Gemini already has: code execution

The tools above are **ours**: we wrote `compound_interest`, and it ran here in our notebook. Gemini also comes with a few **built-in tools** that run on Google's own servers: Google Search, reading a web page, and **code execution**. With code execution there is nothing to define. Gemini writes its own Python, Google runs it, and the model reads the output before answering.

In [ ]:
# Same question, but with Gemini's built-in code tool instead of our function; show_messages shows the code it wrote and ran.
print(ask_gemini(question, tools=[types.Tool(code_execution=types.ToolCodeExecution())],
                 show_usage=True, show_messages=True))

**Question 1.** Compare this answer and its token counts with the `compound_interest` version above. Who wrote the code each time, and where did it run? When would a business prefer its **own** function over letting the model write code?

**Answer**

*Provide your answer here*

### A real MCP server: an agent that acts

So far every tool was either **ours** (`compound_interest`, running in this notebook) or **Google's** (code execution). Neither can be reused anywhere else: to give `compound_interest` to Claude, ChatGPT or a colleague's chatbot, you would copy the code into each one and wire it up again.

The **Model Context Protocol (MCP)** fixes that. It is an open standard: a company puts its tools in a separate program called an **MCP server**, and any app that speaks MCP can connect to it, **discover** what tools it offers, and use them. Nobody writes the connection twice.

GitHub runs an official MCP server. We will hand Gemini its address and have it do something real: **file an issue** (a to-do item) in a GitHub repo of yours. This is the step from a model that *reads* to an agent that *acts*.

Notice how little code it takes. We don't connect to GitHub ourselves: we pass the server's address and our token in the request, and **Google's servers** connect to GitHub, discover its tools, and do the same round trip you just saw with `compound_interest` (Gemini asks for a tool, the tool runs, its result goes back to Gemini), then send back the final answer. This uses Gemini's newer **Interactions API** (`client.interactions.create`). One consequence: your GitHub token travels through Google on its way to GitHub, which is one more reason to give it access to a single repo only.

**Setup (about 4 minutes, free GitHub account is fine).** The token is the agent's key to GitHub, so we give it access to one repo and one kind of action only.

1. On GitHub, create a new **private** repo and tick **Add a README**. Any name works; we use `is883-agent-demo` as the example.
2. Go to **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**.
3. Under **Repository access**, choose **Only select repositories** and pick the repo you just created.
4. Under **Permissions → Repository permissions**, set **Issues** to **Read and write**. Leave everything else alone.
5. Generate the token, copy it, and save it in Colab Secrets (the 🔑 panel) as **`MyGitHubToken`**, with Notebook access switched on.

In [ ]:
# Your GitHub details: change these to your username and the repo you just created.
GITHUB_USER = "your-github-username"   # change to your own GitHub username
REPO = "is883-agent-demo"   # example name; use whatever you called yours

In [ ]:
# Ask Gemini to file an issue. Google's servers connect to GitHub's MCP server, find its tools and use them.
task = (f"Create an issue in the GitHub repo {GITHUB_USER}/{REPO} titled 'Project checkpoint to-dos'. "
        "In the body, list three short to-dos for a student team building a GenAI app. "
        "Then tell me the link to the new issue.")

interaction = client.interactions.create(
    model=MODEL,
    input=task,
    tools=[{
        "type": "mcp_server",
        "name": "github",                                   # any label, no dashes
        "url": "https://api.githubcopilot.com/mcp/x/issues",        # GitHub's server, issue tools only
        "headers": {"Authorization": "Bearer " + get_key("MyGitHubToken")},
        "allowed_tools": [{"mode": "auto", "tools": ["issue_write", "list_issues"]}],   # only these tools
    }],
    store=False,   # don't keep this conversation on Google's servers afterwards
)
print(interaction.output_text)

In [ ]:
# Peek behind the scenes: each step Google's servers took, and the token bill for the whole run.
for step in interaction.steps or []:
    if step.type == "function_call":                      # Gemini asks the MCP server to run a tool
        print("Gemini called:", step.name, "with", step.arguments)
    elif step.type == "function_result":                  # what the MCP server sent back
        print("Server returned:", getattr(step.result, "result", step.result))
if not any(step.type == "function_call" for step in interaction.steps or []):
    print("Gemini did not call any tool this time.")

usage = interaction.usage
print("*" * 20)
print("input_tokens:   ", usage.total_input_tokens)
print("output_tokens:  ", usage.total_output_tokens)
print("thinking_tokens:", usage.total_thought_tokens)
print("tools_tokens:   ", usage.total_tool_use_tokens)
print("total_tokens:   ", usage.total_tokens)
print("*" * 20)

Now open your repo's **Issues** tab (`https://github.com/<your-username>/<your-repo>/issues`). The issue is there, written by Gemini through GitHub's own server.

**Question 2.** This time the model did not just answer; it **changed something** in the world. List everything we did to limit what it *could* do (think about the token, the URL and `allowed_tools`). Your token also passed through Google's servers: what does that add to the risk? What could go wrong if we had used a token with access to all your repos, or connected to an MCP server someone else wrote? And why does it matter to a company that the same GitHub server works unchanged in Claude, Cursor and Gemini?

**Answer**

*Provide your answer here*

### What MCP actually saves you

Is MCP worth it, or is it just a new name for a function call? To find out, we build the same "file an issue" tool **without** MCP, then run two checks.

**The experiment: write the tool ourselves.** No MCP: we read GitHub's web API documentation and write our own Python function that files an issue, the way we built `compound_interest`. It uses the same token.

Watch two things: how much code **you** had to write, and who would have to fix it if GitHub changed something tomorrow.

In [ ]:
# Our own "file an issue" tool, written by hand against GitHub's web API (no MCP).
import requests

def create_github_issue(title: str, body: str) -> str:
    """Create an issue in the class GitHub repo and return its web link.

    Args:
        title: a short title for the issue.
        body: the issue text, for example a list of to-dos.
    """
    response = requests.post(
        f"https://api.github.com/repos/{GITHUB_USER}/{REPO}/issues",
        headers={"Authorization": "Bearer " + get_key("MyGitHubToken"),
                 "Accept": "application/vnd.github+json"},
        json={"title": title, "body": body},
        timeout=30,
    )
    response.raise_for_status()          # stop with an error if GitHub said no
    return response.json()["html_url"]

print(ask_gemini(task.replace("Project checkpoint to-dos", "Project checkpoint to-dos (round 2)"),
                 tools=[create_github_issue], show_messages=True))

Open your repo's Issues tab: you should now see two issues, one filed through GitHub's MCP server and one through our own function.

**Check 1: how many tools did we skip writing?** Our own function is **one** tool, and it took about 15 lines plus a trip through GitHub's API documentation. The next cell asks GitHub's MCP server for its full menu. This is the same "discover the tools" step Google's servers performed for us above, run by hand. (`await` just means "wait for GitHub to answer".)

In [ ]:
# Ask GitHub's MCP server to list its tools: the issues-only menu we gave Gemini, then the full menu.
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

async def list_mcp_tools(url):
    headers = {"Authorization": "Bearer " + get_key("MyGitHubToken")}
    async with create_mcp_http_client(headers=headers) as http:
        async with streamable_http_client(url, http_client=http) as streams:
            async with ClientSession(streams[0], streams[1]) as session:
                await session.initialize()
                return (await session.list_tools()).tools

issue_tools = await list_mcp_tools("https://api.githubcopilot.com/mcp/x/issues")
all_tools = await list_mcp_tools("https://api.githubcopilot.com/mcp/x/all")

print(f"Issues-only address: {len(issue_tools)} tools.  Full address: {len(all_tools)} tools.  Our own code: 1 tool.\n")
for tool in all_tools:
    print(f"- {tool.name}: {(tool.description or '').strip().splitlines()[0][:90]}")

Every tool on that list came with a name, a description and an argument list that the model reads, exactly like the docstring on `compound_interest`. GitHub wrote them, and GitHub updates them when its API changes. Your token still decides which ones can *succeed*: ours only allows issues in one repo, so the rest would be refused.

**Check 2: the same server with no code at all** *(instructor demo, in Claude Code)*. One command in a terminal hands the same server and the same one-repo token to Claude Code:

```
claude mcp add --transport http github https://api.githubcopilot.com/mcp/x/issues -H "Authorization: Bearer <your token>"
```

Then, inside Claude Code: *"File an issue in my repo titled 'Project checkpoint to-dos (round 3)', with three to-dos for a student team."* No Python, no function, no loop. Your `create_github_issue` function cannot go there: it lives only in this notebook. Remove it afterwards with `claude mcp remove github`, because the command saved your token in Claude Code's settings.

Why not the Claude website? GitHub's server needs a sign-in method that Claude's web connectors don't support yet ([GitHub's install guide](https://github.com/github/github-mcp-server/blob/main/docs/installation-guides/install-claude.md)). "Works with any app" still depends on each app supporting the server's sign-in.

| | GitHub's MCP server | Our own function |
| --- | --- | --- |
| Tools you get | everything in Check 1 | 1 |
| Code you wrote | none: an address and a token | about 15 lines, after reading GitHub's API docs |
| Who fixes it when GitHub changes its API | GitHub | you |
| Use it with no programming (Claude Code, VS Code, Cursor) | one command or one click | not possible |
| Extra moving parts | a server someone else runs, one more network hop, and your token passes through Google's servers | none: your token goes straight to GitHub |

The case for MCP is the first four rows: **tools you didn't write, don't maintain, and can hand to people who can't code.**

**When MCP is not worth it:** one small function that only your own app uses. A plain function is simpler, faster to debug, and keeps your token at home. Most project teams' own helper functions are in this group.

**Question 3.** Your team's app needs one outside tool (for example, looking up flights or creating calendar events). Would you use a company's MCP server or write the function yourselves? Use the table: name one reason for each side, and say which one matters more for your project. Then look at the last row: what would make you prefer writing your own?

**Answer**

*Provide your answer here*

# Part 2: Chain-of-Thought vs. ReAct

Last week you met **Chain-of-Thought (CoT)**: ask the model to reason step by step, and it often does better. **ReAct** (*Reasoning + Acting*, Yao et al., 2022 — [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)) also works in steps, so the two are easy to confuse. The difference is **where each step's information comes from**:

| | Chain-of-Thought | ReAct |
|---|---|---|
| What the steps are | The model *writes out* its reasoning | **Thought → Action → Observation**, repeated |
| Where facts come from | Only the model's memory (closed book) | Tools: real lookups and exact calculations (open book) |
| Model calls | One | One per step; the model decides when to stop |
| Can a step use something new? | No: every step is written from what the model already "knows" | Yes: each Thought can depend on the Observation just before it |
| Typical failure | A remembered number is wrong or out of date, and every later step builds on it | A tool fails or returns something unexpected, or the loop wanders |

In short, **CoT breaks a problem into steps of thinking; ReAct breaks it into steps of thinking *and doing*.** Let's run both on a question that needs several facts the model can't know, in a particular order.

In [ ]:
# A multi-step investing question: each step needs a fact (a price) or exact math from the step before.
scenario = ("On January 3, 2022 I bought 10 shares of Microsoft (MSFT). Exactly two years later, on January 3, 2024, "
            "I invested another $5,000 in MSFT at that day's price (fractional shares are fine). "
            "I have held everything until today. How many shares do I own, what are they worth today, "
            "and what is my total gain compared with the money I put in?")

### First, Chain-of-Thought

We ask for step-by-step reasoning and switch on the model's built-in thinking, so you can watch it reason. It has **no tools**: every price must come from memory.

In [ ]:
# CoT: the model reasons step by step, but only from what it remembers.
print(ask_gemini(scenario + " Think step by step and show your working.",
                 thinking=True, include_thoughts=True, show_usage=True))

The reasoning probably *looks* convincing. But look at the inputs: where did each price come from? And what did it say about **today's** price? Keep this answer to compare with in a minute.

### Now, ReAct

We give the model three tools: a **stock-price lookup** (real market data, through the free `yfinance` library), an exact **calculator**, and `get_today` from Part 1.

In [ ]:
# Two new tools for a real investing question: a stock-price lookup and an exact calculator.
import datetime
import yfinance as yf

def get_stock_price(ticker: str, date: str) -> dict:
    """Return a stock's closing price on a date, or on the last trading day before it (for weekends and holidays).

    Args:
        ticker: the stock's ticker symbol, e.g. "MSFT".
        date: the date as YYYY-MM-DD.
    """
    day = datetime.date.fromisoformat(date)
    prices = yf.Ticker(ticker).history(start=day - datetime.timedelta(days=10),
                                       end=day + datetime.timedelta(days=1), auto_adjust=False)
    if prices.empty:
        return {"error": f"No price found for {ticker} near {date}."}
    return {"trading_day": prices.index[-1].date().isoformat(), "close": round(float(prices["Close"].iloc[-1]), 2)}

def calculate(expression: str) -> float:
    """Return the exact result of an arithmetic expression, e.g. "10 * 334.75" or "5000 / 370.6 + 10".

    Args:
        expression: numbers combined with + - * / and parentheses.
    """
    if not set(expression) <= set("0123456789.+-*/() "):
        return "Error: only numbers, + - * / and parentheses are allowed."
    return eval(expression)

Now we hand Gemini the three tools and let the SDK run the loop for us, exactly as in Part 1. `show_messages=True` prints every request behind the scenes, and each request is one ReAct step: the model writes a short **Thought** (what it needs next and why), asks for an **Action** (a tool call), and gets back the **Observation** (the tool's result). The loop ends when the model answers without asking for a tool. Watch for a step whose Action only makes sense because of the Observation right before it: that is what CoT cannot do.

In [ ]:
# ReAct: Gemini runs Thought -> Action -> Observation with our tools until it has the answer.
print(ask_gemini(scenario, tools=[get_stock_price, calculate, get_today],
                 system_instruction=("Before each tool call, write one short sentence saying what you need next and why. "
                                     "Use the calculate tool for every piece of arithmetic."),
                 show_usage=True, show_messages=True))

**Question 4.** Compare the CoT answer with the ReAct answer. Which numbers in the CoT answer could the model not possibly have known, and how far off were they? Find one step in the ReAct trace whose Action depended on the Observation just before it. Why can't Chain-of-Thought do that, however carefully it reasons? Finally, look at the token counts: what does ReAct's extra accuracy cost?

**Answer**

*Provide your answer here*

# Part 3: Embeddings

Tools let the model *act*. Now we give it *knowledge*. The first building block is the **embedding**: a way to turn a piece of text into a list of numbers (a **vector**) that captures its meaning. Texts with similar meaning get similar vectors — even if they share no words.

In [ ]:
# An embedding turns text into a vector of numbers that represents its meaning.
import numpy as np

EMBED_MODEL = "gemini-embedding-001"

def embed(texts):
    """Return a NumPy array of embedding vectors, one row per input text."""
    result = client.models.embed_content(model=EMBED_MODEL, contents=texts)
    return np.array([e.values for e in result.embeddings])

sentences = [
    "The cat sat on the mat.",
    "A kitten rested on the rug.",
    "Our quarterly revenue grew by 12%.",
]
vectors = embed(sentences)
print("Each sentence is now a vector of length", vectors.shape[1])

In [ ]:
# Cosine similarity scores how close two vectors point (1 = same meaning, ~0 = unrelated).
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(vectors)   # every sentence compared with every other one, as a table
print("cat  vs kitten :", round(scores[0, 1], 3))   # different words, same idea
print("cat  vs revenue:", round(scores[0, 2], 3))   # unrelated topics

Notice the first pair scores high even though "cat/mat" and "kitten/rug" share no words — the embedding captured *meaning*, not spelling. That is what makes semantic search possible.

# Part 4: Chunking — and Why It Matters

Real documents are not one sentence — they are pages that cover many topics at once. Here is a realistic example: a single airline policy document covering baggage, refunds, pets, boarding, and Wi-Fi.

In [ ]:
# One realistic document that covers several different topics.
document = """FlyExample Airlines - Customer Policy Summary.

Checked baggage. Every ticket includes one free checked bag weighing up to 50 pounds. Each
additional checked bag costs $35, and any bag over 50 pounds carries a $100 overweight fee.

Refunds and cancellations. Tickets are fully refundable if you cancel at least 24 hours before
departure. Cancellations made within 24 hours receive a travel credit instead of a cash refund.
Refunds are processed within 14 business days to the original form of payment.

Pets. Small pets are welcome in the cabin for a $125 fee and must stay in an approved carrier
that fits under the seat. Larger animals travel in the cargo hold for a $200 fee. Service
animals fly free of charge.

Boarding. Boarding begins 40 minutes before departure and the gate closes 15 minutes before
departure. Passengers who arrive after the gate closes are rebooked on the next flight.

In-flight Wi-Fi. Wi-Fi is free for loyalty program members. Everyone else may buy a full-flight
pass for $8. Streaming video is not supported on the free tier."""

print("The document is", len(document.split()), "words long and covers 5 different topics.")

**The naive idea:** embed the whole document as one vector and search over that. The problem: a single vector for a whole page is a *blurry average* of every topic in it. When someone asks about pets, that average only partly matches — the baggage, refund, and Wi-Fi text drags it away from the pet policy.

**The fix — chunking:** split the document into smaller pieces (**chunks**), embed each chunk on its own, and search the chunks. Now the pet question can match the *pet chunk* directly. Below is a simple chunker. It has two knobs you will tune a lot in practice: how big each chunk is, and how much neighboring chunks **overlap** (so a sentence split across a boundary isn't lost).

In [ ]:
# Split text into overlapping chunks of about `words_per_chunk` words each.
def chunk_text(text, words_per_chunk=40, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + words_per_chunk]))
        start += words_per_chunk - overlap   # step forward, but overlap keeps shared context
    return chunks

chunks = chunk_text(document)
print("Split into", len(chunks), "chunks. Here are the first three:\n")
for chunk in chunks[:3]:
    print("-", chunk, "\n")

Now let's *measure* why chunking helps. We compare how strongly a pet question matches the **whole document** versus its **best single chunk**.

In [ ]:
# Compare retrieval sharpness: whole-document vector vs. the best chunk vector.
query = "How much does it cost to bring my dog on board?"

whole_doc_vector = embed([document])[0]     # one blurry vector for the entire page
chunk_vectors = embed(chunks)               # one focused vector per chunk
query_vector = embed([query])[0]

whole_score = cosine_similarity([query_vector], [whole_doc_vector])[0, 0]
chunk_scores = cosine_similarity([query_vector], chunk_vectors)[0]   # one score per chunk

# First the whole document, then every chunk, then the winner.
print("similarity to the WHOLE document:", round(whole_score, 3), "\n")
for i in range(len(chunks)):
    print(f"similarity to chunk {i + 1}: {chunk_scores[i]:.3f}   ({chunks[i][:60]}...)")

best = int(np.argmax(chunk_scores))
print(f"\nBest match: chunk {best + 1}, score {chunk_scores[best]:.3f} (whole document: {whole_score:.3f})")
print(chunks[best])

The best chunk scores noticeably higher than the whole document, and it is the chunk about pets — even though the question said "dog," not "pet." That is the payoff of chunking: **sharper matches and a smaller, on-topic slice of text** to hand the model.

**Question 5.** Re-run the chunker with a very small size (`chunk_text(document, words_per_chunk=5, overlap=1)`) and a very large one (`words_per_chunk=500`). What goes wrong at each extreme? Relate the large-chunk case back to the "blurry average" problem.

**Answer**

*Provide your answer here*

# Part 5: Retrieval with Cosine Similarity

Retrieval means: given a question, return the chunks whose meaning is closest to it. We already embedded every chunk once above; for each new question we embed it too and keep the highest-scoring chunks. This is a search engine in a few lines of Python.

In [ ]:
# Return the k chunks whose meaning is closest to the query.
def retrieve(query, k=2):
    query_vector = embed([query])[0]
    scores = cosine_similarity([query_vector], chunk_vectors)[0]   # one score per chunk
    ranked = np.argsort(scores)[::-1]          # sort from highest score to lowest
    return [chunks[i] for i in ranked[:k]]

for chunk in retrieve("Can I get my money back if I cancel two days early?"):
    print("-", chunk, "\n")

# Part 6: Retrieval-Augmented Generation (RAG)

Now we combine the two halves. **RAG** = *retrieve* the relevant chunks, then let the model *generate* an answer using only those chunks as context. This is exactly the grounding trick from Session 4 — but now the context is chosen automatically for each question. It keeps answers accurate and lets the model say "I don't know" when the knowledge base doesn't cover something.

In [ ]:
# RAG: retrieve the most relevant chunks, then answer using ONLY those chunks.
def answer_with_rag(question, k=2):
    context = "\n".join(retrieve(question, k))
    prompt = (f"Answer the question using ONLY the context below. "
              f"If the answer is not in the context, say you don't know.\n\n"
              f"Context:\n{context}\n\n"
              f"Question: {question}")
    return ask_gemini(prompt, show_usage=True)

# In the knowledge base:
print(answer_with_rag("How much does it cost to fly with my dog?"))
print("-" * 60)
# NOT in the knowledge base -> the model should admit it doesn't know.
print(answer_with_rag("What is the CEO's salary?"))

**Question 6.** Change one fact in `document` (e.g., make the extra-bag fee $50), re-run the chunking and embedding cells, and ask the baggage question. Does the answer follow your edit? What does that tell you about where a RAG system's "knowledge" actually lives?

**Answer**

*Provide your answer here*

# Part 7: A Business Example — Quarterly Revenue

The airline document made the mechanics clear, but it was invented. Here is the same idea on a question a finance team actually asks, using a document that really exists.

The setup matters: our model has **no internet access and no search tool**. Everything it "knows" was frozen when it finished training. So when we ask about a company's latest quarter, it has exactly two options — repeat the most recent quarter it happened to memorise, or admit it doesn't know. Watch which one it picks.

In [ ]:
# Ask cold: no documents, no internet. Just the model's parameters.
revenue_question = ("What was NVIDIA's total revenue in its most recently reported quarter? "
                    "Give the exact figure and name the quarter.")

print(ask_gemini(revenue_question, show_usage=True))

Whatever number came back, notice two things:

1. **It is stated fluently**, with no warning that it might be out of date.
2. **It cannot possibly cover a quarter reported after the model finished training.** NVIDIA's revenue has been growing roughly 20% *per quarter*, so an answer one year stale is not slightly off — it is off by tens of billions of dollars.

This is the failure mode that gets people in trouble. The model is not lying and it is not broken; it is answering the only way it can.

Our knowledge base is the real thing: NVIDIA's quarterly earnings press releases, downloaded straight from its newsroom ([nvidianews.nvidia.com](https://nvidianews.nvidia.com)). We keep only each page's headline and article text, not the menus around it.

In [ ]:
# Download NVIDIA's last four quarterly earnings press releases from its newsroom.
import requests
from bs4 import BeautifulSoup

release_urls = [
    "https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2026",
    "https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-fourth-quarter-and-fiscal-2026",
    "https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-first-quarter-fiscal-2027",
    "https://nvidianews.nvidia.com/news/nvidia-announces-financial-results-for-second-quarter-fiscal-2027",
]

def read_press_release(url):
    """Download one press release and return its headline plus its article text (no menus or links)."""
    page = BeautifulSoup(requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text, "html.parser")
    return page.select_one("h1").get_text(strip=True) + ". " + page.select_one("div.article-body").get_text(" ", strip=True)

filings = [read_press_release(url) for url in release_urls]
for url, text in zip(release_urls, filings):
    print(f"{len(text.split()):>5} words  {url}")

Each release runs to about 3,000 words, more than the embedding model reads at once (roughly 2,000 tokens). So we do exactly what Part 4 taught: split each release into chunks and embed every chunk, remembering which release each chunk came from.

In [ ]:
# Split every release into chunks (Part 4's chunker), remember each chunk's source, and embed them all.
filing_chunks, chunk_sources = [], []
for url, text in zip(release_urls, filings):
    for chunk in chunk_text(text, words_per_chunk=300, overlap=50):
        filing_chunks.append(chunk)
        chunk_sources.append(url.split("results-for-")[-1])   # e.g. "first-quarter-fiscal-2027"

filing_vectors = embed(filing_chunks)
print(len(filing_chunks), "chunks embedded from", len(filings), "press releases")

In [ ]:
# Retrieve the closest chunks (and say which release each came from), then answer using only those.
def answer_from_filings(question, k=3):
    question_vector = embed([question])[0]
    scores = cosine_similarity([question_vector], filing_vectors)[0]   # one score per chunk
    best = np.argsort(scores)[::-1][:k]                                 # the k highest scores

    for i in best:
        print(f"Retrieved (similarity {scores[i]:.3f}) from {chunk_sources[i]}:\n   {filing_chunks[i][:150]}...\n")

    context = "\n\n".join(f"[{chunk_sources[i]}] {filing_chunks[i]}" for i in best)
    prompt = (f"Answer the question using ONLY the context below. Quote the figure exactly. "
              f"If the answer is not in the context, say you don't know.\n\n"
              f"Context:\n{context}\n\n"
              f"Question: {question}")
    return ask_gemini(prompt, show_usage=True)

print(answer_from_filings("What was NVIDIA's revenue in the first quarter of fiscal 2027?"))

Same model. Same API. Same question, essentially. The only thing that changed is **what was in the prompt** — and now the answer is both correct and checkable against a document you can point a client to.

### The catch: retrieval can be wrong too

Notice we had to name the quarter — "first quarter of fiscal 2027" — for that to work. Try the vaguer, more natural phrasing instead.

In [ ]:
# The way a person would actually ask it.
print(answer_from_filings("How much revenue did NVIDIA make last quarter?"))

Look at *which* press releases the chunks came from — and re-run the cell a couple of times with slightly different wordings ("this past quarter", "the latest quarter"). Sometimes the newest release wins; often it doesn't. That inconsistency **is** the lesson: the phrase "last quarter" contains no date, so there is nothing in it for the embedding to match a date against. Chunks from all four releases look roughly equally relevant, and whichever one edges ahead does so for reasons that have nothing to do with recency. The model then answers faithfully **from whatever document it was handed** — sounding exactly as confident as it did when it was right.

This is worth sitting with, because it is the failure that actually happens in production. RAG moves the problem rather than removing it: instead of trusting the model's memory, you are now trusting your retrieval. When a RAG system gives a wrong answer, the model is usually fine — the retriever handed it the wrong page. Fixing it is a data problem (store each report's date as metadata and filter on it before ranking by similarity), not a prompting problem.

**Question 7.** Ask `answer_from_filings` a question whose answer is genuinely *not* in any of the four press releases — for example, NVIDIA's revenue two years from now, or its CEO's compensation. What does it do? Then explain, in two or three sentences, why a company might prefer this behaviour to a model that answers from memory.

**Answer**

*Provide your answer here*

### Optional: scaling up with a vector database (Chroma)
We stored our chunk vectors in a NumPy array and compared them ourselves. That is perfect for learning and for small collections. For thousands or millions of chunks, a **vector database** like [Chroma](https://www.trychroma.com/) stores the embeddings and does the similarity search for you — but the idea is identical to what we just built.

**Experiment (optional).** Uncomment the cell below and run it. Chroma retrieves the pet policy the same way our NumPy `retrieve` did — the concept is identical, just handled by a library.

In [ ]:
# Optional: the same retrieval using the Chroma vector database (see the note above to run it).
# !pip install chromadb
# import chromadb
# collection = chromadb.Client().create_collection(name="airline")
# collection.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])
# hits = collection.query(query_texts=["How much to bring my dog?"], n_results=2)
# print(hits["documents"])

# Summary

- **Tools / function calling**: give the model Python functions so it can act (and do exact math).
- **Built-in tools**: Gemini can also write and run its own code (or search, or read a page) on Google's servers, with nothing to define.
- **MCP**: connect to a server someone else runs (GitHub's), discover its tools, and let the agent **act**, with a token limited to exactly what it needs.
- **ReAct** (Yao et al., 2022): thought -> action -> observation -> repeat, with the *model* choosing each tool and when to stop. Unlike Chain-of-Thought, which reasons only from memory, each step can use a fact the previous step just looked up.
- **Embeddings**: turn text into vectors that capture meaning.
- **Chunking**: split long documents so each vector is focused, not a blurry average of many topics.
- **Retrieval (cosine similarity)**: find the chunks closest in meaning to a question, with scikit-learn's `cosine_similarity`.
- **RAG**: retrieve relevant chunks, then answer using only those — accurate, and honest about gaps.
- **Where RAG breaks**: a vague question retrieves the wrong document, and the model answers confidently from it. RAG replaces "trust the model's memory" with "trust your retrieval."

Together, tools and retrieval turn a text predictor into something that can *act in the world* and *reason over your data*.